In [4]:
import os
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import shap
import joblib

data_path = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/RFSOC_v3_1020_reoder_v3_FesSta_FD2_withoutNewCattle_reorganize_1114final.csv"
whitelist_path = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/dsm_cluster_vif_outputs/kept_vars.txt"

out_dir = "./rf_4nodes_CFval_kfold5_ESM2_cv"

os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.join(out_dir, "figs"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "shap"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "final_model"), exist_ok=True)

target_col = "ESM 3 SOC (t/ha)"
random_state = 42

NODE_NAME = "CFplus_FLSPDgraz"

SHAP_TOPK = 50

def rpiq(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    iqr = np.subtract(*np.nanpercentile(y_true, [75, 25]))
    rmse = np.sqrt(np.nanmean((y_true - y_pred) ** 2))
    return (iqr / rmse) if rmse > 0 else np.inf


def pooled_metrics(y_true, y_pred):
    from sklearn.metrics import r2_score, mean_squared_error

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rp = rpiq(y_true, y_pred)
    return r2, rmse, rp


def resolve_cols(wanted, actual_cols):
    amap = {c.lower(): c for c in actual_cols}
    resolved, missing = [], []
    for w in wanted:
        key = w.lower().strip()
        if key in amap:
            resolved.append(amap[key])
        else:
            missing.append(w)
    return resolved, missing


def make_scaler(scaler_type="standard"):
    return MinMaxScaler() if scaler_type == "minmax" else StandardScaler()


def apply_scaling_and_save(X_train, feature_names, save_json_path, scaler_type="standard"):
    scaler = make_scaler(scaler_type)
    X_train_s = scaler.fit_transform(X_train)

    stats = {"scaler_type": scaler_type, "features": list(feature_names)}
    if scaler_type == "minmax":
        stats.update(
            {
                "min_": scaler.min_.tolist(),
                "scale_": scaler.scale_.tolist(),
                "data_min_": scaler.data_min_.tolist(),
                "data_max_": scaler.data_max_.tolist(),
                "data_range_": scaler.data_range_.tolist(),
            }
        )
    else:
        stats.update(
            {
                "mean_": scaler.mean_.tolist(),
                "scale_": scaler.scale_.tolist(),
                "var_": scaler.var_.tolist(),
            }
        )

    with open(save_json_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return X_train_s

def _mean_encode_series(train_s: pd.Series, train_y: pd.Series, alpha=10.0, prior=None):
    if prior is None:
        prior = float(train_y.mean())
    stats = pd.DataFrame({"cat": train_s.values, "y": train_y.values})
    grp = stats.groupby("cat")["y"].agg(["mean", "count"])
    enc = (grp["count"] * grp["mean"] + alpha * prior) / (grp["count"] + alpha)
    return enc.to_dict(), prior


def target_encode_fit_transform_oof(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cat_cols,
    n_splits=5,
    alpha=10.0,
    random_state=42,
    groups=None,
):
    X_train = X_train.copy()
    y_train = y_train.reset_index(drop=True)
    cat_cols = [c for c in cat_cols if c in X_train.columns]

    for c in cat_cols:
        X_train[c] = X_train[c].astype("category")

    num_cols = [c for c in X_train.columns if c not in cat_cols]

    X_train[num_cols] = X_train[num_cols].replace([np.inf, -np.inf], np.nan)
    train_medians = X_train[num_cols].median(numeric_only=True)
    X_train[num_cols] = X_train[num_cols].fillna(train_medians)

    global_prior = float(y_train.mean())

    for c in cat_cols:
        X_train[f"__te__{c}"] = np.nan

    use_group = (groups is not None) and (pd.Series(groups).nunique() >= n_splits)
    if use_group:
        splitter = GroupKFold(n_splits=n_splits)
        split_iter = list(splitter.split(X_train, y_train, groups=groups))
    else:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = list(splitter.split(X_train))

    # OOF TE
    for (tr_idx, va_idx) in split_iter:
        tr_df, va_df = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        tr_y = y_train.iloc[tr_idx]
        for c in cat_cols:
            mapping, prior = _mean_encode_series(tr_df[c], tr_y, alpha=alpha, prior=global_prior)
            vals = va_df[c].map(mapping).astype(float).fillna(prior)
            X_train.iloc[va_idx, X_train.columns.get_loc(f"__te__{c}")] = vals.to_numpy()

    for c in cat_cols:
        X_train[f"__te__{c}"] = X_train[f"__te__{c}"].fillna(global_prior)

    encoders = {}
    for c in cat_cols:
        mapping, prior = _mean_encode_series(X_train[c], y_train, alpha=alpha, prior=global_prior)
        encoders[c] = {"mapping": mapping, "prior": prior, "alpha": alpha}

    te_cols = [f"__te__{c}" for c in cat_cols]
    enc_feature_names = list(num_cols) + te_cols

    X_train_enc = pd.concat(
        [X_train[num_cols].astype(float), X_train[te_cols].astype(float)], axis=1
    ).values

    return X_train_enc, enc_feature_names, encoders, train_medians.to_dict(), cat_cols, num_cols

def get_farm_series(df: pd.DataFrame, idx):
    farm_col = None
    for cand in ["Farm", "farm", "FARM"]:
        if cand in df.columns:
            farm_col = cand
            break
    if farm_col is None:
        return None
    s = df.loc[idx, farm_col].astype(object)
    s = s.where(s.notna(), "__NA__")
    return s

print("🔹 Step 1: 读取数据 & 清洗列名 …")

data = pd.read_csv(data_path)
data.columns = (
    data.columns.str.replace(r"[\u00A0\u2007\u202F]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

if target_col not in data.columns:
    raise ValueError(f"缺少目标列: {target_col}")

y_all = data[target_col].astype(float)

with open(whitelist_path, "r", encoding="utf-8") as f:
    whitelist = [ln.strip() for ln in f if ln.strip()]

whitelist = [c for c in whitelist if c in data.columns]
if not whitelist:
    raise ValueError("白名单与数据无交集")

user_cat_cols = [
    "Lookup_Soil_Taxnomy_Order",
    "cropLandcover",
    "Annual_NLCD_LndCov_2008_CU_C1V1",
    "Level4_Ecoregions_FL",
    "Surficial_Geology_of_Florida",
    "Lookup_Hydrologic_Soil_Group",
]
present_cat, _missing_cat = resolve_cols(user_cat_cols, data.columns)

feature_cols = list(dict.fromkeys(whitelist + present_cat))
X_full = data[feature_cols].copy()

for c in present_cat:
    if c in X_full.columns:
        X_full[c] = X_full[c].astype("category")

print("🔹 Step 2: 构造 CF/FLSPD 及 grazing 索引 …")

is_cf = data["data_source"].astype(str).str.lower().eq("cattle farm")
is_flspd = data["data_source"].astype(str).str.lower().eq("flspd")
is_graz = data["Gtype_inter"].astype(str).str.lower().eq("grazing")
is_non_graz = data["Gtype_inter"].astype(str).str.lower().eq("no_graz")

idx_CF_graz = np.where(is_cf & is_graz)[0]
idx_FLSPD_graz = np.where(is_flspd & is_graz)[0]
idx_FLSPD_non = np.where(is_flspd & is_non_graz)[0]

if len(idx_CF_graz) < 5:
    raise ValueError(f"CF∩grazing 样本过少（{len(idx_CF_graz)}），之前 5 折会报错。")

train_idx_final = np.sort(np.unique(np.concatenate([idx_CF_graz, idx_FLSPD_graz])))
print(f"  - CF_graz 样本数: {len(idx_CF_graz)}")
print(f"  - FLSPD_graz 样本数: {len(idx_FLSPD_graz)}")
print(f"  - 最终训练样本数（CFplus_FLSPDgraz）: {len(train_idx_final)}")

X_train_raw = X_full.loc[train_idx_final].copy()
y_train = y_all.loc[train_idx_final].copy()

print("🔹 Step 3: 从 rf_4nodes_results.csv 中读取最佳 fold 信息 …")

results_path = os.path.join(out_dir, "rf_4nodes_results.csv")
if not os.path.exists(results_path):
    raise FileNotFoundError(f"未找到结果文件: {results_path}")

res_df = pd.read_csv(results_path)

if "node" not in res_df.columns or "rmse" not in res_df.columns or "fold" not in res_df.columns:
    raise ValueError("rf_4nodes_results.csv 缺少必要列（node / rmse / fold）")

sub = res_df[res_df["node"] == NODE_NAME].copy()
if sub.empty:
    raise ValueError(f"在 rf_4nodes_results.csv 中没有找到节点 {NODE_NAME} 的记录")

best_row = sub.loc[sub["rpiq"].idxmax()]
best_fold = int(best_row["fold"])
best_params_json = best_row["best_params"]
DO_SCALING = bool(best_row.get("scaled", 1))  # 0/1 -> bool
SCALER_TYPE = str(best_row.get("scaler_type", "standard"))

best_params = json.loads(best_params_json)

print(f"✅ 选定节点: {NODE_NAME}")
print(f"✅ RMSE 最小的 fold: {best_fold}")
print(f"✅ 该 fold 使用的缩放: DO_SCALING = {DO_SCALING}, SCALER_TYPE = '{SCALER_TYPE}'")
print(f"✅ 最优超参数（best_params）:")
print(json.dumps(best_params, indent=2, ensure_ascii=False))

print("🔹 Step 4: 载入 RFECV 选择特征 …")

rfecv_txt = os.path.join(out_dir, f"rfecv_selected_{NODE_NAME}_fold{best_fold}.txt")
if not os.path.exists(rfecv_txt):
    raise FileNotFoundError(f"找不到 RFECV 特征文件: {rfecv_txt}")

with open(rfecv_txt, "r", encoding="utf-8") as f:
    rfecv_selected_features = [ln.strip() for ln in f if ln.strip()]

if not rfecv_selected_features:
    raise ValueError("RFECV 选择特征列表为空，请检查 rfecv_selected_*.txt")

print(f"✅ RFECV 选中特征数: {len(rfecv_selected_features)}")
print("✅ RFECV 选中特征名称示例（前 20 个）:")
for name in rfecv_selected_features[:20]:
    print("   ", name)

print("🔹 Step 5: 对 CFplus_FLSPDgraz 全体训练集做 Target Encoding …")

farm_train = get_farm_series(data, train_idx_final)
groups_for_te = farm_train.values if (farm_train is not None and farm_train.nunique() >= 5) else None

X_train_te, enc_feature_names, encoders, train_medians, cat_cols, num_cols = target_encode_fit_transform_oof(
    X_train_raw,
    y_train,
    cat_cols=present_cat,
    n_splits=5,
    alpha=10.0,
    random_state=1234,
    groups=groups_for_te,
)

print(f"  - TE 后特征总数: {len(enc_feature_names)}")

all_features_txt = os.path.join(
    out_dir, "final_model", f"enc_feature_names_full_order_FINAL_{NODE_NAME}.txt"
)
with open(all_features_txt, "w", encoding="utf-8") as f:
    for nm in enc_feature_names:
        f.write(nm + "\n")
print(f"✅ 已保存 TE 后完整特征顺序列表：{all_features_txt}")

print("🔹 Step 6: 缩放特征 …")

if DO_SCALING:
    scaler_json = os.path.join(
        out_dir, "final_model", f"scaler_{SCALER_TYPE}_{NODE_NAME}_FINAL.json"
    )
    X_train_used = apply_scaling_and_save(
        X_train_te,
        enc_feature_names,
        scaler_json,
        scaler_type=SCALER_TYPE,
    )
    print(f"  - 缩放参数已保存: {scaler_json}")
else:
    X_train_used = X_train_te
    print("  - 未进行缩放（DO_SCALING=False）")

print("🔹 Step 7: 根据 RFECV 选择特征进行子集选择 …")

enc_name_set = set(enc_feature_names)
rfecv_selected_filtered = [nm for nm in rfecv_selected_features if nm in enc_name_set]

if not rfecv_selected_filtered:
    raise ValueError("RFECV 选中特征与当前 enc_feature_names 无交集，请检查！")

if len(rfecv_selected_filtered) < len(rfecv_selected_features):
    missing = [nm for nm in rfecv_selected_features if nm not in enc_name_set]
    print("⚠️ 警告：以下 RFECV 特征现在没有出现在 enc_feature_names 中，将被忽略：")
    for nm in missing:
        print("   -", nm)

sel_indices = [enc_feature_names.index(nm) for nm in rfecv_selected_filtered]
X_train_sel = X_train_used[:, sel_indices]
selected_names_final = [enc_feature_names[i] for i in sel_indices]

print(f"  - 最终用于训练的特征数: {len(selected_names_final)}")
print("  - 按顺序的完整特征列表（selected features）如下：")
for nm in selected_names_final:
    print("     ", nm)

selected_features_txt = os.path.join(
    out_dir, "final_model", f"features_selected_order_FINAL_{NODE_NAME}.txt"
)
with open(selected_features_txt, "w", encoding="utf-8") as f:
    for nm in selected_names_final:
        f.write(nm + "\n")
print(f"✅ 已保存最终 selected 特征顺序列表：{selected_features_txt}")

print("🔹 Step 8: 训练最终 RandomForest 模型 …")

rf_final = RandomForestRegressor(
    random_state=random_state,
    n_jobs=-1,
    **best_params,
)
rf_final.fit(X_train_sel, y_train.values)

y_pred_train = rf_final.predict(X_train_sel)
r2_tr, rmse_tr, rpiq_tr = pooled_metrics(y_train.values, y_pred_train)
print(f"✅ Final RF (CFplus_FLSPDgraz 全体训练) 训练集表现：")
print(f"   - R²   = {r2_tr:.3f}")
print(f"   - RMSE = {rmse_tr:.3f}")
print(f"   - RPIQ = {rpiq_tr:.3f}")

model_path = os.path.join(out_dir, "final_model", f"rf_final_model_{NODE_NAME}.joblib")
joblib.dump(
    {
        "model": rf_final,
        "selected_features": selected_names_final,
        "enc_feature_names_full": enc_feature_names,
        "encoders": encoders,
        "train_medians": train_medians,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "DO_SCALING": DO_SCALING,
        "SCALER_TYPE": SCALER_TYPE,
        "best_params": best_params,
        "train_idx": train_idx_final,
    },
    model_path,
)
print(f"✅ 最终模型已保存：{model_path}")

print("🔹 Step 9: 计算 SHAP（全体训练集） …")

try:
    explainer = shap.TreeExplainer(rf_final, feature_names=selected_names_final)
    shap_vals = explainer.shap_values(X_train_sel)
    base_val = explainer.expected_value

    mean_abs = np.abs(shap_vals).mean(axis=0)
    shap_df = pd.DataFrame(
        {
            "feature": selected_names_final,
            "mean_abs_shap": mean_abs,
        }
    ).sort_values("mean_abs_shap", ascending=False)

    shap_csv = os.path.join(out_dir, "shap", f"shap_meanabs_FINAL_{NODE_NAME}.csv")
    shap_df.to_csv(shap_csv, index=False)
    print(f"✅ SHAP mean|SHAP| 排序结果已保存：{shap_csv}")

    shap_npz = os.path.join(out_dir, "shap", f"shap_full_FINAL_{NODE_NAME}.npz")
    np.savez_compressed(
        shap_npz,
        shap_values=shap_vals,
        base_value=np.array([base_val]),
        feature_names=np.array(selected_names_final),
        X_train_sel=X_train_sel,
        train_idx=train_idx_final,
        y_train=y_train.values,
    )
    print(f"✅ 完整 SHAP 结果已保存（包含 shap_values、X_train_sel 等）：{shap_npz}")

    topk = min(SHAP_TOPK, len(selected_names_final))
    topk_idx = np.argsort(-mean_abs)[:topk]

    exp = shap.Explanation(
        values=shap_vals[:, topk_idx],
        base_values=np.full(X_train_sel.shape[0], base_val),
        data=X_train_sel[:, topk_idx],
        feature_names=[selected_names_final[i] for i in topk_idx],
    )

    plt.figure(figsize=(9, 6.5), dpi=200)
    shap.plots.beeswarm(exp, max_display=topk, show=False)
    plt.title(f"SHAP Beeswarm — {NODE_NAME} (Final model, train set)", fontsize=11)
    plt.tight_layout()
    beeswarm_png = os.path.join(
        out_dir, "shap", f"shap_beeswarm_FINAL_{NODE_NAME}.png"
    )
    plt.savefig(beeswarm_png)
    plt.close()
    print(f"✅ SHAP beeswarm 图已保存：{beeswarm_png}")

except Exception as e:
    print(f"⚠️ SHAP 计算或绘图失败：{e}")

print("\n🎉 All Done: 最终模型训练 + 完整 SHAP 保存 + 特征顺序输出完成。")


🔹 Step 1: 读取数据 & 清洗列名 …
🔹 Step 2: 构造 CF/FLSPD 及 grazing 索引 …
  - CF_graz 样本数: 75
  - FLSPD_graz 样本数: 154
  - 最终训练样本数（CFplus_FLSPDgraz）: 229
🔹 Step 3: 从 rf_4nodes_results.csv 中读取最佳 fold 信息 …
✅ 选定节点: CFplus_FLSPDgraz
✅ RMSE 最小的 fold: 4
✅ 该 fold 使用的缩放: DO_SCALING = True, SCALER_TYPE = 'standard'
✅ 最优超参数（best_params）:
{
  "max_depth": 20,
  "max_features": 1.0,
  "min_samples_leaf": 1,
  "min_samples_split": 2,
  "n_estimators": 200
}
🔹 Step 4: 载入 RFECV 选择特征 …
✅ RFECV 选中特征数: 32
✅ RFECV 选中特征名称示例（前 20 个）:
    SM_sma_anomaly_sum
    tmean_sma_anomaly_sum
    NDVI_fourier2_phase_rad
    gpw
    vpdmax_trend_slope_per_month
    vpdmin_cv
    tdmean_trend_slope_per_month
    vpdmin_autocorr_lag1
    SM_fourier1_amplitude
    SM_autocorr_lag1
    EVI_cv
    EVI_range
    NDVI_range
    vpdmin_trend_slope_per_month
    vpdmin_range
    PPT_integral_sum
    wc2.1_30s_srad_08
    tmean_interannual_cv
    tdmean_autocorr_lag1
    wc2.1_30s_bio_8
🔹 Step 5: 对 CFplus_FLSPDgraz 全体训练集做 Target Encoding …
 

In [5]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from quantile_forest import RandomForestQuantileRegressor

import joblib

DATA_CSV = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/RFSOC_v3_1020_reoder_v3_FesSta_FD2_withoutNewCattle_reorganize_1114final.csv"

OUT_DIR = "./rf_4nodes_CFval_kfold5_ESM2_cv"
NODE_NAME = "CFplus_FLSPDgraz"

MODEL_JOBLIB = os.path.join(OUT_DIR, "final_model", f"rf_final_model_{NODE_NAME}.joblib")

SCALER_JSON_RF = os.path.join(OUT_DIR, "final_model",
                              f"scaler_standard_{NODE_NAME}_FINAL.json")

SELECTED_FEATURES_TXT = os.path.join(
    OUT_DIR, "final_model", f"features_selected_order_FINAL_{NODE_NAME}.txt"
)

IN_FEATURE_TIF = "/home/ji.song/orange/Jiayi/Grazing_SOC_Feature_TIF_v2/tif_set/composite_32bands.tif"

OUT_PRED_TIF = "soc_pred_mean_q05_q95.tif"

TARGET_COL = "ESM 3 SOC (t/ha)"
RANDOM_STATE = 42
Q_LOW = 5    
Q_HIGH = 95  
NODATA_OUT = -9999.0   

def load_rf_package(joblib_path):
    pkg = joblib.load(joblib_path)
    return pkg

def load_scaling_stats_for_rf(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        stats = json.load(f)

    scaler_type = stats.get("scaler_type", "standard")
    feat_names = stats["features"]

    scaling_info = {"scaler_type": scaler_type, "stats": {}}

    if scaler_type == "minmax":
        data_min = np.array(stats["data_min_"], dtype=float)
        data_range = np.array(stats["data_range_"], dtype=float)
        for i, name in enumerate(feat_names):
            scaling_info["stats"][name] = {
                "data_min": float(data_min[i]),
                "data_range": float(data_range[i]),
            }
    else:  # standard
        mean = np.array(stats["mean_"], dtype=float)
        scale = np.array(stats["scale_"], dtype=float)
        var = np.array(stats["var_"], dtype=float)
        for i, name in enumerate(feat_names):
            scaling_info["stats"][name] = {
                "mean": float(mean[i]),
                "scale": float(scale[i]),
                "var": float(var[i]),
            }
    return scaling_info

def scale_block_for_rf(X_block_raw, selected_features, scaling_info):

    if scaling_info is None:
        return X_block_raw

    scaler_type = scaling_info["scaler_type"]
    stats = scaling_info["stats"]

    X_scaled = X_block_raw.astype(np.float32).copy()
    n_features = X_scaled.shape[1]

    for j in range(n_features):
        feat_name = selected_features[j]
        if feat_name not in stats:
            continue

        s = stats[feat_name]
        if scaler_type == "minmax":
            data_min = s["data_min"]
            data_range = s["data_range"]
            if data_range == 0:
                X_scaled[:, j] = 0.0
            else:
                X_scaled[:, j] = (X_scaled[:, j] - data_min) / data_range
        else:
            mean = s["mean"]
            scale = s["scale"]
            if scale == 0:
                X_scaled[:, j] = 0.0
            else:
                X_scaled[:, j] = (X_scaled[:, j] - mean) / scale

    return X_scaled

def build_qrf_training_data(
    data_csv, target_col, rf_pkg, selected_features, node_name="CFplus_FLSPDgraz"
):

    print("🔹 [QRF] 从 CSV 重建训练数据 …")
    df = pd.read_csv(data_csv)
    df.columns = (
        df.columns.str.replace(r"[\u00A0\u2007\u202F]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    if target_col not in df.columns:
        raise ValueError(f"在 CSV 中找不到目标列 {target_col}")

    y_all = df[target_col].astype(float)

    is_cf = df["data_source"].astype(str).str.lower().eq("cattle farm")
    is_flspd = df["data_source"].astype(str).str.lower().eq("flspd")
    is_graz = df["Gtype_inter"].astype(str).str.lower().eq("grazing")

    idx_CF_graz = np.where(is_cf & is_graz)[0]
    idx_FLSPD_graz = np.where(is_flspd & is_graz)[0]

    train_idx = np.sort(np.unique(np.concatenate([idx_CF_graz, idx_FLSPD_graz])))
    print(f"  - CF_graz 样本数: {len(idx_CF_graz)}")
    print(f"  - FLSPD_graz 样本数: {len(idx_FLSPD_graz)}")
    print(f"  - QRF 训练样本总数: {len(train_idx)}")

    encoders = rf_pkg["encoders"]  # {cat_col: {"mapping":..., "prior":...}}
    X_train = pd.DataFrame(index=train_idx)

    for feat in selected_features:
        if feat.startswith("__te__"):
            base_col = feat.replace("__te__", "", 1)
            if base_col not in df.columns:
                raise ValueError(f"TE 特征 {feat} 对应的原始列 {base_col} 不在 CSV 中")
            if base_col not in encoders:
                raise ValueError(f"encoders 中找不到 {base_col} 的编码信息")

            mapping = encoders[base_col]["mapping"]
            prior = encoders[base_col]["prior"]

            s_raw = df.loc[train_idx, base_col]

            keys = list(mapping.keys())
            if len(keys) > 0:
                key_type = type(keys[0])
                if key_type is str:
                    s_cast = s_raw.astype(str)
                else:
                    s_cast = s_raw.astype(key_type)
            else:
                s_cast = s_raw

            vals = s_cast.map(mapping).astype(float).fillna(prior)
            X_train[feat] = vals.values
        else:
            if feat not in df.columns:
                raise ValueError(f"连续特征 {feat} 不在 CSV 中")
            X_train[feat] = df.loc[train_idx, feat].astype(float).values

    y_train = y_all.loc[train_idx].values

    print("🔹 [QRF] 训练特征矩阵形状:", X_train.shape)
    return X_train.values.astype(np.float32), y_train.astype(np.float32)

def apply_te_on_raster_block(
    X_block_raw, selected_features, encoders, te_feature_names
):
    if not te_feature_names:
        return X_block_raw

    X_new = X_block_raw.copy()
    n_pix, n_feats = X_new.shape

    for te_feat in te_feature_names:
        j = selected_features.index(te_feat)
        base_col = te_feat.replace("__te__", "", 1)

        if base_col not in encoders:
            raise ValueError(f"encoders 中找不到 {base_col} 的编码信息，无法对 {te_feat} 做 TE")

        mapping = encoders[base_col]["mapping"]
        prior = encoders[base_col]["prior"]

        vals_raw = X_new[:, j]  
        s = pd.Series(vals_raw)

        keys = list(mapping.keys())
        if len(keys) > 0 and isinstance(keys[0], str):
            s_cast = s.astype(str)
        else:
            s_cast = s

        vals_te = s_cast.map(mapping).astype(float).fillna(prior).to_numpy()
        X_new[:, j] = vals_te.astype(np.float32)

    return X_new

def main():
    print("=== Step 1: 加载 RF 模型与特征顺序 ===")
    rf_pkg = load_rf_package(MODEL_JOBLIB)
    rf_model: RandomForestRegressor = rf_pkg["model"]
    best_params = rf_pkg["best_params"]
    encoders = rf_pkg["encoders"]
    do_scaling_rf = rf_pkg.get("DO_SCALING", True)
    scaler_type_rf = rf_pkg.get("SCALER_TYPE", "standard")

    print(f"  - RF DO_SCALING = {do_scaling_rf}, SCALER_TYPE = {scaler_type_rf}")
    print(f"  - RF best_params = {best_params}")

    with open(SELECTED_FEATURES_TXT, "r", encoding="utf-8") as f:
        selected_features = [ln.strip() for ln in f if ln.strip()]
    n_feats = len(selected_features)
    print(f"  - selected feature 数量: {n_feats}")

    te_feature_names = [f for f in selected_features if f.startswith("__te__")]
    print(f"  - TE 特征（__te__XXX）数量: {len(te_feature_names)}")
    if te_feature_names:
        print("    TE features:", te_feature_names)

    scaling_info_rf = None
    if do_scaling_rf and os.path.exists(SCALER_JSON_RF):
        scaling_info_rf = load_scaling_stats_for_rf(SCALER_JSON_RF)
        print(f"  - 已加载 RF scaler JSON: {SCALER_JSON_RF}")
    else:
        print("  - 未找到 RF scaler JSON 或 DO_SCALING=False，将跳过 RF scaling")
        do_scaling_rf = False

    print("\n=== Step 2: 构建 QRF 训练数据 ===")
    X_train_qrf_raw, y_train_qrf = build_qrf_training_data(
        DATA_CSV, TARGET_COL, rf_pkg, selected_features, node_name=NODE_NAME
    )

    if scaler_type_rf == "minmax":
        scaler_qrf = MinMaxScaler()
    else:
        scaler_qrf = StandardScaler()
    scaler_qrf.fit(X_train_qrf_raw)
    X_train_qrf = scaler_qrf.transform(X_train_qrf_raw)
    print(f"  - QRF scaler 类型: {scaler_type_rf}")
    print("  - QRF 训练矩阵形状:", X_train_qrf.shape)

    print("\n=== Step 3: 训练 QRF 模型 ===")
    qrf_kwargs = {
        "random_state": RANDOM_STATE,
        "n_estimators": best_params.get("n_estimators", 400),
        "max_depth": best_params.get("max_depth", None),
        "min_samples_split": best_params.get("min_samples_split", 2),
        "min_samples_leaf": best_params.get("min_samples_leaf", 1),
        "max_features": best_params.get("max_features", 1.0),
        "n_jobs": -1,
    }
    qrf = RandomForestQuantileRegressor(**qrf_kwargs)
    qrf.fit(X_train_qrf, y_train_qrf)
    print("  - QRF 训练完成。")

    print("\n=== Step 4: 对整幅 TIF 做 RF + QRF 预测 ===")
    
    out_dir = os.path.dirname(OUT_PRED_TIF)
    if out_dir != "":
        os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(IN_FEATURE_TIF) as src:
        if src.count != n_feats:
            raise ValueError(
                f"输入 TIF band 数({src.count}) 与 selected_features 数量({n_feats}) 不一致，请检查顺序和数量！"
            )

        profile = src.profile.copy()
        profile.update(
            dtype="float32",
            count=3,
            nodata=NODATA_OUT,
        )

        with rasterio.open(OUT_PRED_TIF, "w", **profile) as dst:
            for ji, window in src.block_windows(1):
                arr = src.read(window=window).astype(np.float32)  # (bands, h, w)
                bands, h, w = arr.shape

                X_block_raw = arr.reshape(bands, -1).T  # (n_pix, n_feats)
                n_pix = X_block_raw.shape[0]

                # nodata mask：任一 band = src.nodata 即认为无效
                if src.nodata is not None:
                    nodata_mask = np.any(arr == src.nodata, axis=0).reshape(-1)
                else:
                    nodata_mask = np.zeros(n_pix, dtype=bool)

                valid_mask = ~nodata_mask

                out_mean_flat = np.full(n_pix, NODATA_OUT, dtype=np.float32)
                out_qlo_flat = np.full(n_pix, NODATA_OUT, dtype=np.float32)
                out_qhi_flat = np.full(n_pix, NODATA_OUT, dtype=np.float32)

                if valid_mask.sum() > 0:
                    X_valid_raw = X_block_raw[valid_mask]
                    X_valid_with_te = apply_te_on_raster_block(
                        X_valid_raw,
                        selected_features,
                        encoders,
                        te_feature_names,
                    )

                    if do_scaling_rf and scaling_info_rf is not None:
                        X_valid_rf = scale_block_for_rf(
                            X_valid_with_te, selected_features, scaling_info_rf
                        )
                    else:
                        X_valid_rf = X_valid_with_te

                    y_mean_valid = rf_model.predict(X_valid_rf).astype(np.float32)

                    X_valid_qrf = scaler_qrf.transform(X_valid_with_te)
                    q_mat = qrf.predict(
                        X_valid_qrf,
                        quantiles=[Q_LOW / 100.0, Q_HIGH / 100.0],
                    )  # (n_valid, 2)
                    y_qlo_valid = q_mat[:, 0].astype(np.float32)
                    y_qhi_valid = q_mat[:, 1].astype(np.float32)

                    out_mean_flat[valid_mask] = y_mean_valid
                    out_qlo_flat[valid_mask] = y_qlo_valid
                    out_qhi_flat[valid_mask] = y_qhi_valid

                out_mean = out_mean_flat.reshape(h, w)
                out_qlo = out_qlo_flat.reshape(h, w)
                out_qhi = out_qhi_flat.reshape(h, w)

                dst.write(out_mean, 1, window=window)
                dst.write(out_qlo, 2, window=window)
                dst.write(out_qhi, 3, window=window)

                print(f"  - 完成 window {ji}, size=({h},{w})")

    print("\n🎉 完成：输出预测 TIF 已生成：", OUT_PRED_TIF)
    print("    Band 1: RF mean 预测")
    print("    Band 2: QRF 5% quantile (lower)")
    print("    Band 3: QRF 95% quantile (upper)")


if __name__ == "__main__":
    main()


=== Step 1: 加载 RF 模型与特征顺序 ===
  - RF DO_SCALING = True, SCALER_TYPE = standard
  - RF best_params = {'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
  - selected feature 数量: 32
  - TE 特征（__te__XXX）数量: 2
    TE features: ['__te__Lookup_Soil_Taxnomy_Order', '__te__Surficial_Geology_of_Florida']
  - 已加载 RF scaler JSON: ./rf_4nodes_CFval_kfold5_ESM2_cv/final_model/scaler_standard_CFplus_FLSPDgraz_FINAL.json

=== Step 2: 构建 QRF 训练数据 ===
🔹 [QRF] 从 CSV 重建训练数据 …
  - CF_graz 样本数: 75
  - FLSPD_graz 样本数: 154
  - QRF 训练样本总数: 229
🔹 [QRF] 训练特征矩阵形状: (229, 32)
  - QRF scaler 类型: standard
  - QRF 训练矩阵形状: (229, 32)

=== Step 3: 训练 QRF 模型 ===
  - QRF 训练完成。

=== Step 4: 对整幅 TIF 做 RF + QRF 预测 ===
  - 完成 window (0, 0), size=(512,512)
  - 完成 window (0, 1), size=(512,512)
  - 完成 window (0, 2), size=(512,512)
  - 完成 window (0, 3), size=(512,512)
  - 完成 window (0, 4), size=(512,512)
  - 完成 window (0, 5), size=(512,512)
  - 完成 window (0, 6), size=(512,512